# Training Count Experiment
This notebook launches a series of training jobs using increasing training sample counts.

The notebook is split into 2 sections
1. Launching all the training jobs for the experiment.
    * Uses a basic monitoring loop to launch a maximum number of jobs concurrently
    * Saves a yaml file with current status of all training jobs, can be used to restart job launching and monitoring in case of error.
2. Collecting and aggregating the results of the training runs.



# Launch Experiment Training Jobs

In [18]:
# Imports
import boto3
import functools
import math
import time
import yaml
import os

from datetime import datetime
from sagemaker.pytorch import PyTorch
from sagemaker import get_execution_role, Session
from sagemaker.utils import name_from_base, base_from_name

In [2]:
'''print( os.getenv('AWS_ROLE') )
print( os.getenv('WANDB_API_KEY'))
print( dir(os.environ) )
print( os.environ.items() )'''

"print( os.getenv('AWS_ROLE') )\nprint( os.getenv('WANDB_API_KEY'))\nprint( dir(os.environ) )\nprint( os.environ.items() )"

## Experiment Parameters

In [19]:
# Used as the base for all training jobs to keep organized
draw = 0
experiment_name = "17k-dataset-cleaned-geometry-xcld-SC-draw-{}".format(draw)
#experiment_name = "test-count-1"

# List of training counts to try
# For our training split file (split_file.json) which consists of 13446 images, a good spread might be [10   256   588  1037  1643  2461  3564  5054  7066  9781 13446]
# training_counts =  [13446] #[ 128,   256 ,  588 , 1037 , 1643 , 2461 , 3564, 13446 ] # [10, 100, 1000, -1] # -1 to use all images. 
# For our training split file (filtered_split_all_data.json) which consists of 8229 images
training_counts =  [128, 256, 512, 1024, 2048, 4096, 8229] # # -1 to use all images. 
training_counts = [-1] # -1 to use all images. 

# Number of training runs per training count.
# Increasing this number will give better idea of performance variability 
# at the cost of more compute
training_jobs_per_count = 1

# Flag to indicate if models should be trained for equivalent iterations
# If True, epochs is scaled per experiment such that all experiments train for the
# same number of iterations as a run with all training data.
normalize_training_iterations = True

# Maximum number of simultaneous training jobs
# This could be limited by compute capacity or account quota limits
max_concurrent_jobs = 20
poll_rate_minutes = 1  # How often to poll if capacity is available in minutes

# WandB Parameters
WANDB_API_KEY = os.getenv('WANDB_API_KEY')  # Add key here. Recommend not storing key in notebook in git.
wandb_project_name = "wandb-test-count-1"

# Bucket to use for model checkpointing
checkpoint_bucket = "mz-temp" or Session().default_bucket()
checkpoint_s3_uri = None

## Model Parameters
Model parameters are shared between experiment runs. For example, the model type, batch size, learning rate, and augmentations.

In [20]:
train_image_uri = "s3://baselabelinginfrastack-alllabeledimages/segmentable-unproc"
train_mask_uri = "s3://baselabelinginfrastack-alllabeledimages/masks2"
train_split_file_uri = "s3://baselabelinginfrastack-alllabeledimages/json_split_files/17k_dataset_cleaned_geometry_xcld_SC_draw_{}.json".format(draw) # split_file.json" #File that will lock in which images will go to train, test, validation datasets.
#train_split_file_uri = "s3://baselabelinginfrastack-alllabeledimages/filtered_split_all_data.json" # split_file.json" #File that will lock in which images will go to train, test, validation datasets.
#train_split_file_uri = "s3://baselabelinginfrastack-alllabeledimages/split_file.json" # split_file.json" #File that will lock in which images will go to train, test, validation datasets.

metadata_s3_uri = ""

train_volume_size = 1 # Increase/decrease as needed to fit the dataset

#"ml.g5.12xlarge" #be sure to increase the batch size
instance_type = "ml.g5.12xlarge" #"ml.g5.12xlarge" #"ml.g4dn.4xlarge" # "ml.g5.12xlarge" #"ml.g4dn.4xlarge"
pytorch_version = "2.2"
python_version = "py310"

# Using spot instances can provide savings, but there is no gaurantee of acquiring an instance for training
# and it can be interrupted at any time.
# For more details see:
# https://docs.aws.amazon.com/sagemaker/latest/dg/model-managed-spot-training.html
use_spot = False 

# Set the role arn here to be used if not using the default sagemaker role
role = os.getenv('AWS_ROLE')

# Number of training epochs
train_epochs = int( 20 * 1.14149 )

# Set the warm pool duration equal to the polling rate.
# Will enable faster initialization of new jobs
warm_pool_duration = poll_rate_minutes * 60

# See the LaunchTrainingJob notebook for more details on hyperparameters below
# These will be shared between all experiment runs
hyperparameters = {
    "num_workers": 16,
    "model_name": "UNet",
    # Note, with Pytorch Lightning, this is a per GPU batch count. PL will create a batch per device
    # behind the scenes. No need to adjust if changing instances with different gpu counts. 
    # Recommend adjust if changes instances that have different GPU memory (e.g., p3.* -> p3dn.* -> p4d.* -> p4de.*)
    "batch_size": 6,
    "learning_rate": 1e-4,
    "lr_schedular": "cosine_warmup",
    "image_mode": "grayscale",
    "split_file": train_split_file_uri,
    "use_zero":"True", # Set this to False is using instance with a single (1) GPU (e.g. ml.g4.4xlarge)
    "use_amp": "True", 
    # Data Augmentation Parameters
    "image_size": 1200,
    "use_random_resize": "True",
    "use_random_rotation": "False",
    "color_jitter": 0.0,
    "center_crop": "True",
    "center_crop_size": 1200,
    "test_image_size": 1200,
    "test_batch_size": 1,
    "project_name": wandb_project_name,
    #"metadata_file": metadata_s3_uri,
}

metric_def = [
    {"Name": "train:loss", "Regex": "train_loss=(.*?)[,\]]"},
    {"Name": "val:loss", "Regex": "val_loss=(.*?)[,\]]"},
    {"Name": "learning_rate", "Regex": "lr=(.*?)[,\]]"},
    {"Name": "val:acc", "Regex": "val_acc=(.*?)[,\]]"},
    {"Name": "val:jaccard", "Regex": "val_jaccard=(.*?)[,\]]"},
    {"Name": "val:dice", "Regex": "val_dice=(.*?)[,\]]"},
    {"Name": "test:loss", "Regex": "test_loss[ ]*[^\x00-\x7F] (.*?)[^\x00-\x7F]"},
    {"Name": "test:acc", "Regex": "test_acc[ ]*[^\x00-\x7F] (.*?)[^\x00-\x7F]"},
    {"Name": "test:jaccard", "Regex": "test_jaccard[ ]*[^\x00-\x7F] (.*?)[^\x00-\x7F]"},
    {"Name": "test:dice", "Regex": "test_dice[ ]*[^\x00-\x7F] (.*?)[^\x00-\x7F]"},
]

In [5]:
#%run ../bin/create_data_splits.py --image_path "s3://image-segmentation-2k/images" --mask_path "s3://image-segmentation-2k/masks" --split_file "s3://image-segmentation-2k/split_file.json"

## Training Job Helper Functions

In [21]:
def launch_training_job(
        job_name: str,
        train_count: int,
        train_image_uri: str,
        train_mask_uri: str,
        instance_type: str,
        train_volume_size: int,
        pytorch_version: str,
        python_version: str,
        role: str,
        hyperparameters: dict,
        train_epochs: int,
        warm_pool_duration: int,
        normalize_iterations: bool,
        max_train_count: int,
        wandb_api_key: str,
        checkpoint_bucket: str,
        use_spot: bool,
        checkpoint_s3_uri: str, 
):
    print( normalize_iterations )
    if normalize_iterations:
        max_iterations = (max_train_count // hyperparameters['batch_size']) * train_epochs
        epochs = math.ceil(max_iterations / (train_count // hyperparameters['batch_size']))
    else:
        epochs = train_epochs
        
    # Update hyperparameters
    hyperparameters["epochs"] = epochs
    hyperparameters["train_count"] = train_count
    hyperparameters["run_name"] = job_name
    
    if role is None:
        role = get_execution_role()

    if checkpoint_s3_uri is None: 
        checkpoint_s3_uri = f"s3://{checkpoint_bucket}/{job_name}/checkpoints"
    
    print( checkpoint_s3_uri )
    print( checkpoint_bucket )
    local_checkpoint_path = "/opt/ml/checkpoints"
    # Launch the training job
    estimator = PyTorch(
        entry_point="train.py",
        # Point to the repo root one level up,
        # this notebook is at {repo}/notebooks/LaunchTrainingJob.ipynb
        source_dir="../src/",  
        role=role,
        framework_version=pytorch_version,
        py_version=python_version,
        instance_count=1,
        instance_type= instance_type,
        train_volume_size=train_volume_size,
        hyperparameters=hyperparameters, 
        input_mode="File",
        metric_definitions=metric_def,
        environment={"WANDB_API_KEY": wandb_api_key},
        keep_alive_period_in_seconds=warm_pool_duration,
        checkpoint_s3_uri=checkpoint_s3_uri,
        checkpoint_local_path=local_checkpoint_path,
        use_spot_instances=use_spot,
        max_run=3*24*60*60,
    )
    estimator.fit(
        {"train_image": train_image_uri, "train_mask": train_mask_uri}, job_name=job_name, wait=False
    )

# Create helper to only need to pass job_name and count
launch_training_job_by_count = functools.partial(
    launch_training_job,
    train_image_uri=train_image_uri,
    train_mask_uri=train_mask_uri,
    instance_type=instance_type,
    train_volume_size=train_volume_size,
    pytorch_version=pytorch_version,
    python_version=python_version,
    role=role,
    hyperparameters=hyperparameters,
    train_epochs=train_epochs,
    warm_pool_duration=warm_pool_duration,
    normalize_iterations=normalize_training_iterations,
    max_train_count=max(training_counts),
    wandb_api_key=WANDB_API_KEY,
    checkpoint_bucket=checkpoint_bucket,
    use_spot=use_spot,
    checkpoint_s3_uri=checkpoint_s3_uri,
)

In [22]:
def get_training_jobs(exp_name: str, start_time: datetime):
    sm = boto3.client("sagemaker")
    jobs_list = sm.list_training_jobs(CreationTimeAfter=start_time, NameContains=exp_name)
    jobs = {
        job["TrainingJobName"]: job["TrainingJobStatus"] for job in jobs_list["TrainingJobSummaries"]
    }
    return jobs

def resume_from_checkpoint(bookkeeping_filename: str):
    with open(bookkeeping_filename, 'r') as fp:
        status = yaml.safe_load(fp)
    jobs_to_run = status["jobs_to_run"]
    jobs_in_progress = status["jobs_in_progress"]
    jobs_completed = status["jobs_completed"]
    start_time = datetime.fromisoformat(status["start_time"])
    experiment_name = status["experiment_name"]
    print( 'jobs in progress' )
    print( jobs_in_progress )
    # Move all in progress jobs back to the jobs_to_run queue
    sm = boto3.client("sagemaker")
    for job_name in jobs_in_progress:
        # Get the count from the prior run's hyperparameters
        job_desc =  sm.describe_training_job(TrainingJobName=job_name)
        jobs_to_run.append({
            "job_name": job_name,
            "train_count": int( job_desc["HyperParameters"]["train_count"] )
        })
    jobs_in_progress = set()
    # Update all jobs_to_run names to ensure no conflicts with existing names
    jobs_to_run = [
        {"job_name": name_from_base(base_from_name(job["job_name"])), "train_count": job["train_count"]}
        for job in jobs_to_run
    ]
    print( jobs_to_run )
    return jobs_to_run, jobs_in_progress, jobs_completed, experiment_name, start_time

## Launch and Monitor Training Jobs

In [23]:
# Create all JobNames and Parameter Configs
# Use SageMaker's name_from_base function to ensure name is 1) less than 63 characters and 2) has a unique ending
jobs_to_run = []
for count in training_counts:
    for ind in range(training_jobs_per_count):
        job_name = f"{experiment_name}-{count}-it{ind}"
        jobs_to_run.append({
            "job_name": name_from_base(job_name, short=True),
            "train_count": count,
        })
print(f"Creating {len(jobs_to_run)} training jobs.")

Creating 1 training jobs.


In [24]:
# Bookkeeping data to keep track of which jobs are training, which are waiting and those that have completed
jobs_in_progress = set()
jobs_completed = []

# Enter the path to an existing status dump file to resume training progress
resume_from_experiment_run = None # "test-count-8-job-progress-2024-04-19T19-30-22.yaml"

if resume_from_experiment_run:
    (
        jobs_to_run,
        jobs_in_progress,
        jobs_completed,
        experiment_name,
        start_time
    ) = resume_from_checkpoint(resume_from_experiment_run)
    bookkeeping_filename = resume_from_experiment_run
else:
    start_time = datetime.now()
    bookkeeping_filename = f"{experiment_name}-job-progress-{start_time.isoformat(timespec='seconds')}.yaml".replace(":", "-")

'''print( 'jobs to run 2' )
print( jobs_to_run )
print( jobs_in_progress )
print( jobs_completed )'''


"print( 'jobs to run 2' )\nprint( jobs_to_run )\nprint( jobs_in_progress )\nprint( jobs_completed )"

In [25]:
print("Kicking off training jobs")
print(f"Bookkeeping file name: {bookkeeping_filename}")
print("Checkpoint bucket: {}".format(checkpoint_bucket))
# Launch up to max concurrent jobs and keep track of progress
failure_detected = False
while (len(jobs_to_run) + len(jobs_in_progress)) > 0:
    # Get list of all jobs associated with this run
    training_jobs = get_training_jobs(experiment_name, start_time)
    timestamp = datetime.now().isoformat(timespec='seconds')
    print(f"{timestamp} - Start - {len(jobs_to_run)} jobs remaining to run. {len(jobs_in_progress)} jobs are running.")
    
    # See if any of active jobs are completed
    for job_name, status in training_jobs.items():
        if job_name in jobs_in_progress and status in {"Failed", "Completed"}:
            if status == "Failed":
                # An error has occurred and we will stop launching jobs
                print(f"ERROR!! Job '{job_name}' has Failed! Stopping experiment")
                failure_detected = True
                break
            # If completed, move to the completed set
            print(f"\t - Job {job_name} has completed!")
            jobs_completed.append(job_name)
            jobs_in_progress.remove(job_name)
    if failure_detected:
        break
    # See what capacity we have left for launching new jobs
    capacity = max_concurrent_jobs - len(jobs_in_progress)
    for _ in range(capacity):
        if len(jobs_to_run) < 1:
            break
        job_kwargs = jobs_to_run.pop()
        launch_training_job_by_count(**job_kwargs)
        jobs_in_progress.add(job_kwargs["job_name"])
        print(f"\t - Launched training job {job_kwargs['job_name']}")

        
    # Save out the current experiment status
    #bookkeeping_uri = f"s3://{checkpoint_bucket}/{bookkeeping_filename}"
    #with open( bookkeeping_uri, 'w') as fp:
    with open(bookkeeping_filename, 'w') as fp:
        status = {
            "experiment_name": experiment_name,
            "start_time": start_time.isoformat(timespec="seconds"),
            "last_updated": datetime.now().isoformat(timespec="seconds"),
            "jobs_to_run": jobs_to_run,
            "jobs_in_progress": list(jobs_in_progress),
            "jobs_completed": jobs_completed,
        }
        yaml.dump(status, fp)
    
        
    print(f"{timestamp} - End - {len(jobs_to_run)} jobs remaining to run. {len(jobs_in_progress)} jobs are running.")
    # Sleep the thread for the specified wait time
    time.sleep(poll_rate_minutes * 60)

if failure_detected:
    print(f"{datetime.now().isoformat(timespec='seconds')} - Failure occurred during a training job. ")
else:
    print(f"{datetime.now().isoformat(timespec='seconds')} - All jobs completed!")
    print(f"Final bookkeeping file: {bookkeeping_filename}")

Kicking off training jobs
Bookkeeping file name: 17k-dataset-cleaned-geometry-xcld-SC-draw-0-job-progress-2025-02-19T12-50-40.yaml
Checkpoint bucket: mz-temp


See: https://sagemaker.readthedocs.io/en/stable/v2.html for details.


2025-02-19T12:50:44 - Start - 1 jobs remaining to run. 0 jobs are running.
True
s3://mz-temp/17k-dataset-cleaned-geometry-xcld-SC-draw-0--1-it0-250219-1250/checkpoints
mz-temp


INFO:sagemaker.estimator:SMDebug Does Not Currently Support                         Distributed Training Jobs With Checkpointing Enabled
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: 17k-dataset-cleaned-geometry-xcld-SC-draw-0--1-it0-250219-1250


	 - Launched training job 17k-dataset-cleaned-geometry-xcld-SC-draw-0--1-it0-250219-1250
2025-02-19T12:50:44 - End - 0 jobs remaining to run. 1 jobs are running.
2025-02-19T12:51:47 - Start - 0 jobs remaining to run. 1 jobs are running.
2025-02-19T12:51:47 - End - 0 jobs remaining to run. 1 jobs are running.
2025-02-19T12:52:48 - Start - 0 jobs remaining to run. 1 jobs are running.
2025-02-19T12:52:48 - End - 0 jobs remaining to run. 1 jobs are running.
2025-02-19T12:53:48 - Start - 0 jobs remaining to run. 1 jobs are running.
2025-02-19T12:53:48 - End - 0 jobs remaining to run. 1 jobs are running.
2025-02-19T12:54:49 - Start - 0 jobs remaining to run. 1 jobs are running.
2025-02-19T12:54:49 - End - 0 jobs remaining to run. 1 jobs are running.
2025-02-19T12:55:49 - Start - 0 jobs remaining to run. 1 jobs are running.
2025-02-19T12:55:49 - End - 0 jobs remaining to run. 1 jobs are running.
2025-02-19T12:56:49 - Start - 0 jobs remaining to run. 1 jobs are running.
2025-02-19T12:56:49 - E

# Aggregate Job Metrics
We will utilize the metrics definition from the training jobs to parse the test metrics that appeared in the logs.

We will utilize the bookkeeping file used during model training to retrieve all the relevant job names.

In [ ]:
# Update seaborn, pandas and matplotlib
!pip install --upgrade seaborn pandas matplotlib

In [ ]:
# Draw visualizations
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

import boto3
import yaml

In [ ]:
# Load the relevant bookkeeping file
#bookkeeping_filename = "test-count-1-job-progress-2024-04-04T16:10:44.yaml"

with open(bookkeeping_filename, "r") as fp:
    status = yaml.safe_load(fp)
jobs_completed = status["jobs_completed"]

In [ ]:
# For each job, just boto3 to get training job info, specifically hyperparameters and metrics
sm = boto3.client("sagemaker")
metrics = []
for index, job_name in enumerate(jobs_completed):
    job_desc = sm.describe_training_job(TrainingJobName=job_name)
    # Get train count
    train_count = job_desc["HyperParameters"]["train_count"]
    # Grab metrics from the FinalMetricDataList
    final_metrics = {item['MetricName']: item["Value"] for item in job_desc["FinalMetricDataList"]}
    metrics.append({
        "train_count": int(train_count),
        # "id": index,
        "acc": final_metrics["test:acc"],
        "dice": final_metrics["test:dice"],
        "jaccard": final_metrics["test:jaccard"]
    })

In [ ]:
# convert metrics to dataframe for convenience
metrics_df = pd.DataFrame(metrics)
fig, axes = plt.subplots(3, 1, sharex=True, squeeze=True, layout="constrained")
for metric_name, ax in zip(["acc", "dice", "jaccard"], axes):
    sns.lineplot(data=metrics_df, x="train_count", y=metric_name, ax=ax)
    ax.set_title(metric_name)
plt.show()

In [ ]:

#Run custom training job. 
if False:
    cp = 's3://mz-temp/test-count-8-13446-it0-240419-1930/checkpoints/'
    job_kwargs = {'job_name': 'test-count-8-13446-it0-240419-1930-5', 'train_count': 13446, 'checkpoint_s3_uri': cp}
    launch_training_job_by_count(**job_kwargs)

